# Client Mode vs Cluster Mode: Deep Comparison and Use Cases

**Module**: 12_advanced_topics  
**Notebook**: 03_cluster_vs_client_mode.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Complete comparison of Client Mode vs Cluster Mode across Standalone, YARN, and Kubernetes.
- **Why does this topic exist?**: Choosing the wrong deploy mode can crash interactive terminals or isolate scheduled batch jobs.
- **What problem does it solve?**: Understanding where the Driver runs and why network latency matters.
- **Where is it used in real-world Data Engineering?**: Configuring Airflow SparkSubmitOperator in enterprise environments.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Deployment modes, cluster architecture, networking |
| **Required depth** | Working knowledge: can choose client or cluster mode and explain the trade-offs |
| **Data Engineer** | Choose the deployment mode for interactive versus production workloads. |
| **Senior Data Engineer** | Diagnose driver connectivity, logging, and dependency issues caused by mode. |
| **Spark specialist** | Reason about driver lifecycle, networking, and resource accounting. |
| **Next topic** | `12_advanced_topics/04_resource_allocation.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

The deployment mode determines where the driver runs. In client mode, the driver runs on the machine that
submitted the job, which is convenient for notebooks and debugging but ties the job to that machine. In cluster
mode, the driver runs inside the cluster, which is more resilient and is the norm for production batch jobs.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Client mode** | Driver on submit host | Interactive use; driver must stay connected |
| **Cluster mode** | Driver in cluster | Production; driver lifecycle managed by the cluster |
| **Driver** | Scheduling and coordination | Hosts SparkSession and the UI |
| **Gateway** | Submit host | Needs network access to the cluster |
| **Logs** | Diagnostics | Location depends on mode |
| **Dependencies** | Jars and files | Must be available to the driver and executors |

### How it works, step by step

1. The submission client builds the application and chooses a mode.
2. In client mode, it starts the driver locally and connects to the cluster manager.
3. In cluster mode, it asks the cluster manager to start a driver container/pod.
4. The driver requests executors and schedules tasks.
5. The client may exit after submission in cluster mode.
6. Logs and the UI are accessed through the cluster manager or a proxy.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Client vs Cluster Mode Architecture
```
CLIENT MODE:
[Submitting Machine (Jupyter / Laptop)]  <--- High Network Latency Risk!
  - Runs DRIVER Process
  - Requires persistent low-latency connection to cluster!
       | RPC
       v
[Remote Cluster (Workers / Executors)]

CLUSTER MODE:
[Submitting Machine] ---> Submits job manifest to Master and DISCONNECTS (Fire & Forget!)
[Remote Cluster]
  - Node 1: Worker launches DRIVER JVM inside cluster!
  - Node 2, 3: Workers launch EXECUTORS!
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Client_Cluster_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
print("Our Docker JupyterLab environment runs in: CLIENT MODE")
print("Driver Host:   ", spark.conf.get("spark.driver.host"))
print("Master URL:    ", spark.conf.get("spark.master"))

---

## Experiment: Deploy Mode Decision Guide

- **1. What are we testing?**: Print decision rules for Client vs Cluster mode.
- **2. What do we expect?**: Client for interactive/dev; Cluster for production.

In [ ]:
print("""
Rule 1: If using JupyterLab, spark-shell, or PySpark REPL -> MUST USE CLIENT MODE.
Rule 2: If using Airflow, cron, or production scheduled ETL -> ALWAYS USE CLUSTER MODE.
""")

**3. What happened?**: Explained architectural deployment rules.  
**4. Why did it happen?**: Client mode binds Driver to submitting machine; Cluster mode decouples submission from execution.  
**5. Practical Engineering Lesson**: Never use client mode for production scheduled jobs over remote VPNs or CI/CD pipelines.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Notebook exploration

- **Industry**: Data science
- **Scenario**: An analyst wants to interact with a Spark cluster from a Jupyter notebook.
- **Problem**: Cluster mode would require a separate driver process and complicate interactive feedback.
- **Why this concept is relevant**: Client mode keeps the driver in the notebook session, so cells can iterate quickly.
- **Implementation idea**: Use client mode for exploration and switch to cluster mode for scheduled production jobs.
- **Expected result**: Fast iteration with a clear boundary to production.
- **Alternative**: Use a managed notebook service with a remote driver.
- **Trade-offs**: A managed service simplifies connectivity but may hide cluster mechanics.

### Use Case 2: Production batch reliability

- **Industry**: Logistics
- **Scenario**: A nightly job fails when the submitting laptop disconnects.
- **Problem**: In client mode, the driver runs on the submit host and the job dies with it.
- **Why this concept is relevant**: Cluster mode runs the driver inside the cluster, independent of the submit host.
- **Implementation idea**: Submit in cluster mode and retrieve logs from the cluster manager.
- **Expected result**: A job that survives submit-host disconnects.
- **Alternative**: Run the client on an always-on gateway.
- **Trade-offs**: A gateway reduces the risk but still leaves a single point of failure.

---

## Business Value

- Reliable production jobs independent of laptops.
- Clear separation between exploration and operations.
- Correct resource accounting for the driver.
- Better log and dependency management.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Use cluster mode for scheduled production jobs.
- Use client mode for interactive notebooks and debugging.
- Ensure the driver has network access to executors.
- Package dependencies so the driver and executors can resolve them.
- Know where driver logs are stored in each mode.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Compare driver locations

**Steps**

1. Submit a small job in client mode and note where the driver runs.
2. Submit the same job in cluster mode and inspect the driver container/pod.
3. Compare log locations and behavior when the client disconnects.

**What to observe**: Client mode ties the driver to the submit host; cluster mode runs it inside the cluster.

**Healthy result**: The mode matches the workload's lifecycle.

**Problematic result**: A production job dies when the submitting session ends.

**Engineering takeaway**: Deployment mode is a reliability decision.

---

## Trade-offs and Alternatives

### Advantages

- Client mode is convenient
- Cluster mode is resilient
- Explicit lifecycle

### Limitations

- Client mode depends on the submit host
- Cluster mode complicates interactive debugging
- Networking differs

### When to use it

- Notebooks (client)
- Production batch (cluster)
- CI jobs (cluster)

### When not to use it

- Production jobs from laptops
- Assuming logs are local in cluster mode
- Missing driver dependencies

### Alternatives inside Spark

- Driver placement
- Cluster manager
- Submission client

### Alternatives outside Spark

- Managed notebook services
- CI/CD runners
- Gateway hosts

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify Lab Mode
**Problem Statement**: What deploy mode is active in this notebook?  
**Input / Context**: Mode

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Cluster Mode Log Access
**Problem Statement**: How do you view driver logs in cluster mode?  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Network Partition Impact
**Problem Statement**: What happens if submitting machine disconnects in client mode?  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The job that died with the laptop

**Scenario**: A nightly job stops when the engineer closes the laptop used to submit it.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check whether the job was submitted in client mode.
2. Look for driver disconnection errors.
3. Resubmit in cluster mode or from an always-on gateway.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify Lab Mode):
```python
print('Client mode (Jupyter hosts Driver; workers host Executors).')
```

#### Solution 2 (Cluster Mode Log Access):
```python
print('Via yarn logs -applicationId <id> or Spark History Server (18080).')
```

#### Solution 3 (Network Partition Impact):
```python
print('Driver process terminates; entire cluster application is aborted immediately.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Client Mode**: Driver on client; required for notebooks.
- **Cluster Mode**: Driver on worker; fire-and-forget for production.

### Essential Technical Terms
- `Deploy Mode`: Client vs Cluster flag.
- `Fire-and-Forget`: Submitting job and disconnecting.

### Common Engineering Mistakes
- **Running client mode over high-latency VPN**: Heartbeat timeouts kill job.

### Interview Questions on This Topic
- **Q: What is the fundamental difference between Client mode and Cluster mode?**
  - *A*: In Client mode, the Driver runs on the submitting machine; in Cluster mode, the Driver runs on a worker node inside the cluster.


### More Interview Questions

- **Q: What is the main difference between client and cluster mode?**
  - *A*: In client mode the driver runs on the submitting machine; in cluster mode the driver runs inside the cluster and is managed by the cluster manager.
- **Q: When is client mode appropriate?**
  - *A*: For interactive exploration, notebooks, and debugging where the driver needs to stay close to the user; production jobs should generally use cluster mode.

### What to Study Next
Open 04_resource_allocation.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')